## Setup

In [1]:
# import necessary libraries
import time
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
# load paths
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.evaluation import calculate_macro_f1
from src.submission import create_submission

DATA_DIR = PROJECT_ROOT / "data"
SUBMISSIONS_DIR = PROJECT_ROOT / "submissions"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SPLIT_PATH = DATA_DIR / "splits" / "shared_validation_split.csv"

print(TRAIN_PATH.exists(), TEST_PATH.exists(), SPLIT_PATH.exists())

True True True


In [3]:
# load raw data + shared split
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

ID_COLUMN = "id"
LABEL_COLUMN = "label"

split = pd.read_csv(SPLIT_PATH)
assert len(split) == len(train_df)

split = split.set_index("row_index")
is_train = (split.loc[train_df.index, "split"] == "train").to_numpy()

X_train_text = train_df.loc[is_train, "text"]
X_val_text = train_df.loc[~is_train, "text"]
y_train = train_df.loc[is_train, LABEL_COLUMN].to_numpy()
y_val = train_df.loc[~is_train, LABEL_COLUMN].to_numpy()

print(X_train_text.shape, X_val_text.shape)
print(round(y_train.mean(), 4), round(y_val.mean(), 4))

(16000,) (4000,)
0.6252 0.6252


In [4]:
# TF-IDF
tfidf = TfidfVectorizer()
X_train = tfidf.fit_transform(X_train_text) 
X_val = tfidf.transform(X_val_text) 

print(X_train.shape, X_val.shape)

(16000, 82595) (4000, 82595)


In [5]:
# char-level TF-IDF
char_tfidf = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=5,
    max_features=3000,
)

X_train_char = char_tfidf.fit_transform(X_train_text)
X_val_char = char_tfidf.transform(X_val_text)

print("char feature count:", X_train_char.shape[1])

char feature count: 3000


In [6]:
# combine both
X_train_combined = hstack([X_train, X_train_char]).tocsr()
X_val_combined = hstack([X_val, X_val_char]).tocsr()

print("combined shape:", X_train_combined.shape, X_val_combined.shape)

combined shape: (16000, 85595) (4000, 85595)


In [15]:
print("word-only shape:", X_train.shape)
print("combined shape:", X_train_combined.shape)
print("char block nonzero sum:", X_train_char.nnz)

word-only shape: (16000, 82595)
combined shape: (16000, 85595)
char block nonzero sum: 11408627


# Complement Naive Bayes (CNB) Model

In [7]:
# build model
def cnb_fit(X, y, alpha=1.0, normalize=True):
    classes = np.unique(y)
    n_classes = len(classes)
    n_features = X.shape[1]

    class_log_prior = np.zeros(n_classes)
    complement_weights = np.zeros((n_classes, n_features))

    total_feature_mass = np.asarray(X.sum(axis=0)).ravel()   # sum across ALL documents, once

    for idx, c in enumerate(classes):
        X_c = X[y == c]
        class_log_prior[idx] = np.log(X_c.shape[0] / X.shape[0])

        class_feature_mass = np.asarray(X_c.sum(axis=0)).ravel()
        complement_mass = total_feature_mass - class_feature_mass + alpha   # all OTHER classes' mass
        complement_total = complement_mass.sum()

        weights = np.log(complement_mass / complement_total)

        if normalize:
            weights = weights / np.sum(np.abs(weights))   # per original CNB paper — improves stability

        complement_weights[idx] = weights

    return classes, class_log_prior, complement_weights

In [8]:
# predict
def cnb_predict_scores(X, class_log_prior, complement_weights):
    return X @ complement_weights.T


def cnb_predict(X, classes, class_log_prior, complement_weights):
    scores = cnb_predict_scores(X, class_log_prior, complement_weights)
    return classes[np.argmin(scores, axis=1)]


def cnb_predict_proba(X, class_log_prior, complement_weights):
    scores = cnb_predict_scores(X, class_log_prior, complement_weights)
    neg_scores = -scores                                  # flip sign so higher = more likely, for softmax
    neg_scores -= neg_scores.max(axis=1, keepdims=True)
    proba = np.exp(neg_scores)
    return proba / proba.sum(axis=1, keepdims=True)

In [9]:
# find best 'alpha' value
cnb_results = []

for alpha in [0.01, 0.1, 0.5, 1, 2]:
    classes, class_log_prior, complement_weights = cnb_fit(X_train, y_train, alpha=alpha)
    preds = cnb_predict(X_val, classes, class_log_prior, complement_weights)
    score = calculate_macro_f1(y_val, preds)
    cnb_results.append({"alpha": alpha, "val_macro_f1": score})
    print(f"alpha={alpha:<6} CNB F1={score:.4f}")

cnb_results_df = pd.DataFrame(cnb_results).sort_values("val_macro_f1", ascending=False).reset_index(drop=True)
cnb_results_df

alpha=0.01   CNB F1=0.5182
alpha=0.1    CNB F1=0.4816
alpha=0.5    CNB F1=0.4030
alpha=1      CNB F1=0.3946
alpha=2      CNB F1=0.3897


,alpha,val_macro_f1
0,0.01,0.518236
1,0.10,0.481587
2,0.50,0.403050
3,1.00,0.394592
4,2.00,0.389675


In [10]:
best_alpha = float(cnb_results_df.iloc[0].alpha)
print("best alpha:", best_alpha, "val Macro F1:", round(float(cnb_results_df.iloc[0].val_macro_f1), 6))

best alpha: 0.01 val Macro F1: 0.518236


In [11]:
# refit TF-IDF + model on labelled data
X_full_text = train_df["text"]
y_full = train_df[LABEL_COLUMN].to_numpy()

final_tfidf = TfidfVectorizer()
X_full = final_tfidf.fit_transform(X_full_text) 

final_classes, final_class_log_prior, final_feature_log_prob = cnb_fit(X_full, y_full, alpha=best_alpha)

In [12]:
# transform test text with final vectoriser
X_test = final_tfidf.transform(test_df["text"])
test_preds = cnb_predict(X_test, final_classes, final_class_log_prior, final_feature_log_prob)

print(test_preds)

[1 1 1 ... 1 1 1]


In [13]:
# save data 
cnb_submission_path = SUBMISSIONS_DIR / "CNB_Prediction.csv"
cnb_submission_df = create_submission(
    test_ids=test_df[ID_COLUMN],
    predictions=test_preds,
    output_path=cnb_submission_path,
    id_column=ID_COLUMN,
    label_column=LABEL_COLUMN,
)
display(cnb_submission_df.head())

,id,label
0,59218,1
1,37110,1
2,23200,1
3,e3357348-166e-4847-a06d-158b7cd83aa5,1
4,61615,1


In [14]:
saved = pd.read_csv(cnb_submission_path, dtype={ID_COLUMN: "string"})
expected_ids = test_df[ID_COLUMN].astype("string").reset_index(drop=True)

assert saved.columns.tolist() == [ID_COLUMN, LABEL_COLUMN]
assert len(saved) == len(test_df)
assert saved[ID_COLUMN].tolist() == expected_ids.tolist()
assert saved[LABEL_COLUMN].isnull().sum() == 0
assert set(saved[LABEL_COLUMN].unique()).issubset(set(train_df[LABEL_COLUMN].unique()))

print("verified — ready to upload")

verified — ready to upload
